In [1]:

from transformers import  AutoTokenizer
import os
from codecarbon import EmissionsTracker
from time import time
import csv
from vllm import LLM, SamplingParams

import wandb

import requests

WARNING 09-23 15:10:43 cuda.py:22] You are using a deprecated `pynvml` package. Please install `nvidia-ml-py` instead, and make sure to uninstall `pynvml`. When both of them are installed, `pynvml` will take precedence and cause errors. See https://pypi.org/project/pynvml for more information.


In [2]:
%env WANDB_LOG_MODEL=true
%env WANDB_WATCH=all

env: WANDB_LOG_MODEL=true
env: WANDB_WATCH=all


In [3]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [4]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    CBLACKBG  = '\33[40m'
    CREDBG    = '\33[41m'
    CGREENBG  = '\33[42m'
    CYELLOWBG = '\33[43m'
    CBLUEBG   = '\33[44m'
    CVIOLETBG = '\33[45m'
    CBEIGEBG  = '\33[46m'
    CWHITEBG  = '\33[47m'
    CBLACK  = '\33[30m'
    CRED    = '\33[31m'
    CGREEN  = '\33[32m'
    CYELLOW = '\33[33m'
    CBLUE   = '\33[34m'
    CVIOLET = '\33[35m'
    CBEIGE  = '\33[36m'
    CWHITE  = '\33[37m'

# Preparing the Test Data

In [5]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.bos_token

In [6]:
# Laden des Textes von der URL
url = "https://www.gutenberg.org/cache/epub/2701/pg2701.txt"
response = requests.get(url)
text = response.text

In [7]:

def prepare_prompts(text, token_length, runs=1, sum_sentences=3):
    system_message = f"""
    You are an AI assistant designed to summarize a book.
    Generate {sum_sentences} short and precise sentence(s) that summarize the given text.

    For example, a summary for a short story:
    'The story follows a young boy who discovers a hidden talent. He faces challenges but overcomes them with determination. In the end, he achieves his dream and inspires others.'

    Be as close as possible to the example summary!
    Respond with only the {sum_sentences} sentence(s) itself. Do not include any introductory phrases.
    Your response should be as short as possible with exactly {sum_sentences} sentences.

    """

    prompts = []
    
    # Entfernen der ersten 1000 Wörter
    words = text.split()
    if len(words) > 1519:
        truncated_text = ' '.join(words[1519:])
    else:
        truncated_text = ' '.join(words)
    
    # Wiederholen des Textes, bis er länger als token_length ist
    while len(truncated_text.split()) < token_length:
        truncated_text += ' ' + truncated_text
    
    # Trunkieren des Textes auf die gewünschte Länge
    truncated_text = ' '.join(truncated_text.split()[:token_length])
    
    for r in range(runs):
        user_message = f"Question: Now, summarize the following text in {sum_sentences} short and precise sentences:\n{truncated_text}\nAnswer:"
        
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )

        prompts.append(prompt)
    
    return prompts

In [8]:

def showcase_promp(text, token_length, runs=1, sum_sentences=3):
    system_message = f"""
    You are an AI assistant designed to summarize a book.
    Generate {sum_sentences} short and precise sentence(s) that summarize the given text.

    For example, a summary for a short story:
    'The story follows a young boy who discovers a hidden talent. He faces challenges but overcomes them with determination. In the end, he achieves his dream and inspires others.'

    Be as close as possible to the example summary!
    Respond with only the {sum_sentences} sentence(s) itself. Do not include any introductory phrases.
    Your response should be as short as possible with exactly {sum_sentences} sentences.

    """
    
    # Entfernen der ersten 1000 Wörter
    words = text.split()
    if len(words) > 1519:
        truncated_text = ' '.join(words[1519:])
    else:
        truncated_text = ' '.join(words)
    
    # Wiederholen des Textes, bis er länger als token_length ist
    while len(truncated_text.split()) < token_length:
        truncated_text += ' ' + truncated_text
    
    # Trunkieren des Textes auf die gewünschte Länge
    truncated_text = ' '.join(truncated_text.split()[:token_length])
    
    for r in range(runs):
        user_message = f"Question: Now, summarize the following text in {sum_sentences} short and precise sentences:\n{truncated_text}\nAnswer:"
        
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ]

    
    return messages

# vLLM

## Creating the Model

In [9]:
# Create an LLM.
llm = LLM(model=model_name,
          tensor_parallel_size=4, 
          dtype='bfloat16'
          )

INFO 09-23 15:10:47 config.py:904] Defaulting to use mp for distributed inference
WARNING 09-23 15:10:47 arg_utils.py:900] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 09-23 15:10:47 config.py:1013] Chunked prefill is enabled with max_num_batched_tokens=512.
INFO 09-23 15:10:47 llm_engine.py:223] Initializing an LLM engine (v0.6.1.post2) with config: model='meta-llama/Meta-Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=4, pipeline_parallel_size

WARNING 09-23 15:10:47 multiproc_gpu_executor.py:56] Reducing Torch parallelism from 24 threads to 1 to avoid unnecessary CPU contention. Set OMP_NUM_THREADS in the external environment to tune this value as needed.
INFO 09-23 15:10:47 custom_cache_manager.py:17] Setting Triton cache manager to: vllm.triton_utils.custom_cache_manager:CustomCacheManager
(VllmWorkerProcess pid=106634) (VllmWorkerProcess pid=106635) (VllmWorkerProcess pid=106633) INFO 09-23 15:10:48 multiproc_worker_utils.py:215] Worker ready; awaiting tasks
INFO 09-23 15:10:48 multiproc_worker_utils.py:215] Worker ready; awaiting tasks
INFO 09-23 15:10:48 multiproc_worker_utils.py:215] Worker ready; awaiting tasks
INFO 09-23 15:10:49 utils.py:981] Found nccl from library libnccl.so.2
INFO 09-23 15:10:49 pynccl.py:63] vLLM is using nccl==2.20.5
(VllmWorkerProcess pid=106635) (VllmWorkerProcess pid=106634) INFO 09-23 15:10:49 utils.py:981] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=106633) INFO 09-23 15:10

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(VllmWorkerProcess pid=106635) INFO 09-23 15:10:51 model_runner.py:1008] Loading model weights took 3.7710 GB
INFO 09-23 15:10:51 model_runner.py:1008] Loading model weights took 3.7710 GB
(VllmWorkerProcess pid=106633) INFO 09-23 15:10:52 model_runner.py:1008] Loading model weights took 3.7710 GB
(VllmWorkerProcess pid=106634) INFO 09-23 15:10:52 model_runner.py:1008] Loading model weights took 3.7710 GB
INFO 09-23 15:10:53 distributed_gpu_executor.py:57] # GPU blocks: 29528, # CPU blocks: 8192
(VllmWorkerProcess pid=106634) INFO 09-23 15:10:56 model_runner.py:1311] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
(VllmWorkerProcess pid=106634) INFO 09-23 15:10:56 model_runner.py:1315] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager

## Testing the Model

In [10]:
def query_model_vllm(prompt_list, temperature=0.2, min_p=0.05, frequency_penalty=1.25, presence_penalty=0.75, max_tokens=250):

    # Create a sampling params object.
    sampling_params = SamplingParams(temperature=temperature, min_p=min_p, frequency_penalty=frequency_penalty, presence_penalty=presence_penalty, max_tokens=max_tokens)

    # Start Timer for Inference
    start_time = time()

    outputs = llm.generate(prompt_list, sampling_params)

    # End Timer for Inference
    end_time = time()

    ttime = end_time-start_time

    return outputs, ttime

### Test Run

In [11]:
def print_prompt(prompt, completion, with_system=True):
    print("="*30 + f" Chat with  --- {model_name} ---  LLM using vLLM " + "="*30 + "\n")
    for idx, message in enumerate(prompt):
        if prompt[idx]['role'] == 'user':
            color = bcolors.CBLUE
            print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
            print(prompt[idx]['content']+ "\n")
        else: 
            if with_system:
                color = bcolors.CVIOLET
                print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
                print(prompt[idx]['content']+ "\n")

    print(bcolors.OKGREEN + f"[ system ]" + bcolors.ENDC)
    print(completion)

In [12]:
prompts = prepare_prompts(text, token_length=500, runs=1, sum_sentences=3)

prompt_showcase = showcase_promp(text, token_length=500, runs=1, sum_sentences=3)

outputs, ttime = query_model_vllm(prompts)



Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it, est. speed input: 653.02 toks/s, output: 45.95 toks/s]


In [13]:
print_prompt(prompt_showcase, outputs[0].outputs[0].text)

============================== Chat with  --- meta-llama/Meta-Llama-3.1-8B-Instruct ---  LLM using vLLM ==============================

[ SYSTEM ]

    You are an AI assistant designed to summarize a book.
    Generate 3 short and precise sentence(s) that summarize the given text.

    For example, a summary for a short story:
    'The story follows a young boy who discovers a hidden talent. He faces challenges but overcomes them with determination. In the end, he achieves his dream and inspires others.'

    Be as close as possible to the example summary!
    Respond with only the 3 sentence(s) itself. Do not include any introductory phrases.
    Your response should be as short as possible with exactly 3 sentences.

    

[ USER ]
Question: Now, summarize the following text in 3 short and precise sentences:
“Scarcely had we proceeded two days on the sea, when about sunrise a great many Whales and other monsters of the sea, appeared. Among the former, one was of a most monstrous size.

### Benchmark

In [14]:
# Token-Längen und Anzahl der Prompts
token_lengths = [ 50, 100, 250, 500, 1000, 2500, 5000, 7500, 10000, 15000]
num_prompts = 1000

runs = len(token_lengths)

total_prompts = runs * num_prompts


total_input_tok = 0
total_output_tok = 0

print("="*10 + f" INFERENCE TEST with {num_prompts} PROMPTS " + "="*10 + 
"\n\n" + 
f"""
Starting Test with {runs} Runs and {num_prompts} Prompts / Run. \n
Total Prompts: {total_prompts}\n\n
""")




for tok_length in token_lengths:

    name=f"vLLM_{tok_length}_word_summary"

    prompts = prepare_prompts(text, tok_length, runs=num_prompts, sum_sentences=3)

    wandb.init(
        # set the wandb project where this run will be logged
        project="Inference_Framework_Comparison",

        # track hyperparameters and run metadata
        config={
        "runs": runs,
        "num_prompts": num_prompts,
        "total_prompts": total_prompts,
        "framework": 'vLLM',
        "model": model_name,
        },

        name=name,
    )

    tracker = EmissionsTracker(save_to_file=True, project_name=f"{name}-llama3.1-8B", log_level="error", pue = 1.22, output_file=f"input_tok_summary_vllm.csv")
    tracker.start()


    outputs, ttime = query_model_vllm(prompts)

    emissions: float = tracker.stop()



    for output in outputs: 


        # Extracting information
        prompt = output.prompt
        generated_text = output.outputs[0].text
        input_tokens = output.prompt_token_ids
        output_tokens = output.outputs[0].token_ids
        num_input_tokens = len(input_tokens)
        num_output_tokens = len(output_tokens)

        # Updating cumulative counts
        total_input_tok += num_input_tokens
        total_output_tok += num_output_tokens


    # Calculate averages
    avg_time_per_prompt = (ttime / num_prompts)*1000
    avg_toks_per_sec = total_output_tok/ttime
    avg_input_tokens = total_input_tok / num_prompts
    avg_output_tokens = total_output_tok / num_prompts

    em_i = emissions/total_input_tok *1_000_000
    em_o = emissions/total_output_tok *1_000_000
    em_p = emissions/total_prompts *10_000

    print("="*15 + f" RESULTS for {name} " + "="*15 + 
        "\n\n" + 
        f"""
        Finished {runs} Runs with {num_prompts} Prompts/Run.\n\n
        Total Time: {ttime:.2f}s, AVG/Prompt: {avg_time_per_prompt:.2f}ms\n\n
        Average tokens per second: {avg_toks_per_sec:.2f}\n\n
        Total Prompts: {total_prompts}\n
        Total Input Tokens: {total_input_tok}, AVG/Prompt: {avg_input_tokens}\n
        Total Output Tokens: {total_output_tok}, AVG/Prompt: {avg_output_tokens}\n
        """ + 
        
        "-"*50 + "\n" +
        
        f"""
        Total Inference Emissions: {emissions:.3f}kg CO₂eq\n\n
        Emissions / 1.000.000 Input Tokens: {em_i:.3f}kg CO₂eq\n
        Emissions / 1.000.000 Output Tokens: {em_o:.3f}kg CO₂eq\n
        Emissions / 10.000 Prompts: {em_p:.3f}kg CO₂eq\n

        """
        )

    wandb.log({"Total Time": ttime,
        "AVG. Time / Prompt": avg_time_per_prompt,
                "AVG. Tokens / Second": avg_toks_per_sec,
                "AVG. Input Tokens": avg_input_tokens,
                "AVG. Output Tokens": avg_output_tokens,
                "Total Emissions": emissions,
                "Emissions / 1.000.000 Input Tokens": em_i,
                "Emissions / 1.000.000 Output Tokens": em_o,
                "Emissions / 10.000 Prompts": em_p,
                })

    wandb.finish()

    # Save results to a CSV file
    results = [
        ["Runs", runs],
        ["Prompts / Run", num_prompts],
        ["Total Prompts", total_prompts],
        ["Total Time", ttime], 
        ["AVG. Time / Prompt", avg_time_per_prompt],
        ["AVG. Tokens / Second", avg_toks_per_sec],
        ["Total Input Tokens", total_input_tok],
        ["AVG. Input Tokens / Prompt", avg_input_tokens],
        ["Total Output Tokens", total_output_tok],
        ["AVG. Output Tokens / Prompt", avg_output_tokens],
        ["Total Emissions", emissions],
        ["Emissions / 1.000.000 Input Tokens", em_i],
        ["Emissions / 1.000.000 Output Tokens", em_o],
        ["Emissions / 10.000 Prompts", em_p]
    ]

    # Ensure the directory exists
    output_file_path = f"emission_data/vllm_input_tok_summary/{name}.csv"
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

    with open(output_file_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Metric", "Value"])
        writer.writerows(results)

    print(f"Results saved to {output_file_path}\n\n")

========== INFERENCE TEST with 1000 PROMPTS ==========


Starting Test with 10 Runs and 1000 Prompts / Run. 

Total Prompts: 10000





wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.


wandb: Currently logged in as: daniel-wetzel (llm-emissions). Use `wandb login --relogin` to force relogin


ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/hardware/cpu_power.csv


Processed prompts: 100%|██████████| 1000/1000 [01:04<00:00, 15.49it/s, est. speed input: 3143.63 toks/s, output: 717.86 toks/s]
/opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/output_methods/file.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame.from_records([dict(total.values)])])


...

Processed prompts:   2%|▎         | 25/1000 [01:40<54:31,  3.36s/it, est. speed input: 3458.63 toks/s, output: 21.22 toks/s]  

ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json


Processed prompts:   6%|▌         | 57/1000 [03:39<1:13:21,  4.67s/it, est. speed input: 3610.78 toks/s, output: 21.56 toks/s]

ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json


Processed prompts:   6%|▌         | 60/1000 [03:44<42:02,  2.68s/it, est. speed input: 3709.73 toks/s, output: 22.17 toks/s]  

# Note: Idle Performance

- In idle each L4 GPU needs about 27W to store its maximum capacity in VRAM. 
- In full idle with empty VRAM each L4 needs about 16W

In [11]:
import time

tracker = EmissionsTracker(save_to_file=True, project_name=f"idle_vllm_output", log_level="error", pue = 1.22, output_file=f"emissions_output_tok_vllm.csv")
tracker.start()


time.sleep(60)

tracker.stop()

/opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/output_methods/file.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame.from_records([dict(total.values)])])


0.0031655554134173997